# DS5 Step 2 Path A — Behavior-Only Phenotyping

**Stage:** Stage 4, Step 2 — Path A  
**Date:** 2026-04-19  
**Blueprint:** `outputs/reports/stage4/ds5_path_a_behavior_only_blueprint.md`  
**Notebook:** `102_ds5_behavior_only_phenotype.ipynb`

## Purpose

Test whether a genuinely different phenotype structure emerges from behavioral dynamics
features alone (sig_hat, H_emp, x_mean) when all network topology features are excluded.

Path C showed that the Phase 1 k=2 split is almost perfectly reproduced by NeT features
alone (ARI=0.990). Behavioral features are nearly identical across all four Path C
sub-profiles. Path A tests whether these same behavioral features have their own grouping
structure when the NeT dimension is not present.

## Step 2 boundary (mandatory)

- No Step 3 personalization or intervention prescription
- No condition used as clustering feature
- Raw H_out excluded (condition-confounded); H_out_resid used for overlay only
- NeT features used ONLY as post-hoc comparison labels — never in clustering
- All findings are exploratory and DS5-specific

In [ ]:
# ─── 0. Setup ──────────────────────────────────────────────────────────────────
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA
from scipy.stats import chi2_contingency, kruskal

ROOT        = Path('..')
PHENOTYPING = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
CLEANED     = ROOT / 'outputs' / 'cleaned'
FIG_DIR     = PHENOTYPING / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_BOOTSTRAPS = 100
SAMPLE_FRAC  = 0.80
MIN_CLUSTER  = 30
K_RANGE      = [2, 3, 4]

# Core 3-feature set (minimal, clean, no NaN)
CORE_FEATURES = ['sig_hat', 'H_emp', 'x_mean']
# 4-feature sensitivity run
SENS_FEATURES = ['sig_hat', 'H_emp', 'x_mean', 'H_hour']

print('ROOT resolved to:', ROOT.resolve())
print('Figures directory:', FIG_DIR.resolve())

## 1. Load data

In [ ]:
# ─── 1. Load data ──────────────────────────────────────────────────────────────

# Feature matrix (all behavior + NeT features; NeT excluded from clustering)
feat = pd.read_csv(PHENOTYPING / 'ds5_phenotype_feature_matrix.csv')

# Phase 1 cluster assignments (post-hoc overlay)
p1 = pd.read_csv(PHENOTYPING / 'ds5_cluster_assignments.csv')[[
    'participant_id_canonical', 'profile_p1', 'cluster_p1'
]]

# Path C sub-cluster assignments (post-hoc ARI comparison)
net_sub = pd.read_csv(PHENOTYPING / 'ds5_net_subcluster_assignments.csv')[[
    'participant_id_canonical', 'profile_p1', 'subprofile_B', 'subprofile_A',
    'cluster_net_only'
]]

# Psychosocial scales (per-participant)
psy = pd.read_csv(CLEANED / 'ds5_pre_survey_clean.csv')[[
    'participant_id_canonical', 'fomo_mean', 'belong_mean', 'ostracism_mean',
    'digital_wellbeing', 'social_media_wellbeing'
]]

# Merge everything
df = feat.merge(p1, on='participant_id_canonical', how='left')
df = df.merge(net_sub[['participant_id_canonical','subprofile_B','subprofile_A',
                        'cluster_net_only']],
              on='participant_id_canonical', how='left')
df = df.merge(psy, on='participant_id_canonical', how='left')

# Create composite Path C label (B1, B2, A1, A2)
df['subprofile_C'] = df['subprofile_B'].fillna(df['subprofile_A'])

# Verify completeness of core features
for col in CORE_FEATURES:
    assert df[col].isna().sum() == 0, f"{col} has NaN values!"

print(f'N total: {len(df)}')
print(f'Phase 1 profile distribution:')
print(df['profile_p1'].value_counts())
print(f'\nPath C subprofile distribution:')
print(df['subprofile_C'].value_counts())
print(f'\nCore features complete: {(df[CORE_FEATURES].notna().all(axis=1)).sum()} / {len(df)}')
print(f'H_hour NaN: {df["H_hour"].isna().sum()}')
print(f'mu_hat NaN: {df["mu_hat"].isna().sum()}')
print(f'sigma_mu_ratio NaN: {df["sigma_mu_ratio"].isna().sum()} ({df["sigma_mu_ratio"].isna().mean()*100:.1f}%)')

## 2. Step 1 — Descriptive: behavior-only feature space inspection

In [ ]:
# ─── 2a. Feature distributions and correlation matrix ─────────────────────────

print('=== Univariate statistics: core + secondary features ===')
feat_inspect = CORE_FEATURES + ['H_hour', 'mu_hat', 'sigma_mu_ratio']
print(df[feat_inspect].describe().round(4))

print('\n=== Correlation matrix: core + H_hour ===')
corr_cols = CORE_FEATURES + ['H_hour']
corr = df[corr_cols].corr().round(3)
print(corr)

print('\n=== H_hour–sig_hat correlation note ===')
r_hh_sig = df['H_hour'].corr(df['sig_hat'])
print(f'  r(H_hour, sig_hat) = {r_hh_sig:.3f} '
      f'— strong negative: high noise → lower temporal spread')
print(f'  This means H_hour partially duplicates sig_hat when both included in clustering.')

print('\n=== mu_hat near-zero check ===')
mu_near_zero = (df['mu_hat'].abs() < 0.01).mean()
print(f'  |mu_hat| < 0.01: {mu_near_zero*100:.1f}% of participants')
print(f'  → mu_hat excluded from clustering; used as overlay only')

In [ ]:
# ─── 2b. Figure A1: Histograms of core + H_hour features ─────────────────────

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('DS5 Path A — Behavior-only feature distributions (N=414)', fontsize=14)

feature_labels = {
    'sig_hat':  'sig_hat (OU noise / diffusion)',
    'H_emp':    'H_emp (behavioral entropy)',
    'x_mean':   'x_mean (mean override rate)',
    'H_hour':   'H_hour (temporal entropy; 24-hour spread)'
}
axes_flat = axes.flatten()

for i, (feat_name, label) in enumerate(feature_labels.items()):
    ax = axes_flat[i]
    ax.hist(df[feat_name].dropna(), bins=40, color='steelblue', edgecolor='white',
            alpha=0.85)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(feat_name)
    ax.set_ylabel('Count')
    mean_val = df[feat_name].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.2,
               label=f'mean={mean_val:.3f}')
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a1_behav_only_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a1_behav_only_distributions.png')

In [ ]:
# ─── 2c. Figure A2: Pairplot with Phase 1 profile overlay ─────────────────────

plot_df = df[CORE_FEATURES + ['H_hour', 'profile_p1']].dropna()
pal = {'Profile A': '#e67e22', 'Profile B': '#2980b9'}

g = sns.pairplot(
    plot_df,
    vars=CORE_FEATURES + ['H_hour'],
    hue='profile_p1',
    palette=pal,
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 20},
    height=2.5
)
g.fig.suptitle('DS5 Path A — Pairplot colored by Phase 1 profile\n'
               '(no NeT features — behavior-only space)',
               fontsize=12, y=1.02)
g.fig.savefig(FIG_DIR / 'fig_a2_behav_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a2_behav_pairplot.png')
print()
print('=== Behavior feature means by Phase 1 profile ===')
print(df.groupby('profile_p1')[CORE_FEATURES + ['H_hour']].mean().round(4))

In [ ]:
# ─── 2d. Figure A3: PCA biplot on core 3 features ─────────────────────────────

scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(df[CORE_FEATURES])

pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_pca_scaled)

print('=== PCA explained variance (core 3 features) ===')
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ev*100:.1f}%  (cumulative: {pca.explained_variance_ratio_[:i+1].sum()*100:.1f}%)')

print('\n=== PCA loadings ===')
loadings_df = pd.DataFrame(
    pca.components_.T,
    index=CORE_FEATURES,
    columns=['PC1', 'PC2', 'PC3']
)
print(loadings_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: PC1 vs PC2 colored by Phase 1 profile
for profile, grp in df.groupby('profile_p1'):
    idx = grp.index
    axes[0].scatter(X_pca[idx, 0], X_pca[idx, 1],
                    alpha=0.4, s=18, label=profile,
                    color=pal[profile])

# Loadings arrows
scale_factor = 2.5
for feat_name, (pc1_load, pc2_load, _) in zip(CORE_FEATURES, pca.components_.T):
    axes[0].annotate('', xy=(pc1_load*scale_factor, pc2_load*scale_factor),
                     xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    axes[0].text(pc1_load*scale_factor*1.15, pc2_load*scale_factor*1.15,
                 feat_name, fontsize=9)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title('PCA biplot: PC1 vs PC2\n(colored by Phase 1 profile)')
axes[0].legend()
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(0, color='gray', linewidth=0.5)

# Panel 2: scree plot
axes[1].bar(['PC1', 'PC2', 'PC3'],
            pca.explained_variance_ratio_ * 100,
            color='steelblue', edgecolor='white')
axes[1].set_ylabel('Explained variance (%)')
axes[1].set_title('PCA scree plot (core 3 features)')
for i, ev in enumerate(pca.explained_variance_ratio_):
    axes[1].text(i, ev*100 + 0.5, f'{ev*100:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a3_behav_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a3_behav_pca.png')

## 3. Step 2 — Feature scaling

In [ ]:
# ─── 3. Feature scaling ─────────────────────────────────────────────────────────

# Primary: 3-feature core
scaler_3 = StandardScaler()
X3 = scaler_3.fit_transform(df[CORE_FEATURES].values)

# Sensitivity: 4-feature (includes H_hour)
scaler_4 = StandardScaler()
X4 = scaler_4.fit_transform(df[SENS_FEATURES].values)

print('Primary feature matrix (3-feature) shape:', X3.shape)
print('Sensitivity feature matrix (4-feature) shape:', X4.shape)
print('\nScaled means (should all be ~0.0):')
print(pd.DataFrame(X3, columns=CORE_FEATURES).mean().round(6))
print('\nScaled stds (should all be ~1.0):')
print(pd.DataFrame(X3, columns=CORE_FEATURES).std().round(6))

## 4. Step 3 — Cluster selection on core 3-feature set

In [ ]:
# ─── 4a. Bootstrap Jaccard stability function ──────────────────────────────────

def bootstrap_jaccard_stability(X, k, n_bootstraps=100, sample_frac=0.8, random_state=42):
    """Pairwise co-membership Jaccard stability across bootstrap resamples."""
    rng = np.random.RandomState(random_state)
    n = len(X)
    sample_n = int(n * sample_frac)

    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_state)
    labels_full = km_full.fit_predict(X)

    jaccard_scores = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=sample_n, replace=False)
        X_boot = X[idx]
        km_boot = KMeans(n_clusters=k, init='k-means++', n_init=10,
                         random_state=int(rng.randint(10000)))
        labels_boot = km_boot.fit_predict(X_boot)

        L_full = labels_full[idx]
        L_boot = labels_boot
        tril_i, tril_j = np.tril_indices(sample_n, k=-1)
        sf = L_full[tril_i] == L_full[tril_j]
        sb = L_boot[tril_i] == L_boot[tril_j]
        tp = (sf & sb).sum()
        fp = (~sf & sb).sum()
        fn = (sf & ~sb).sum()
        denom = tp + fp + fn
        jaccard_scores.append(tp / denom if denom > 0 else 0.0)

    return float(np.mean(jaccard_scores)), float(np.std(jaccard_scores))

In [ ]:
# ─── 4b. K-selection: 3-feature primary ───────────────────────────────────────

results_3 = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X3)
    sil  = silhouette_score(X3, labels)
    ch   = calinski_harabasz_score(X3, labels)
    db   = davies_bouldin_score(X3, labels)
    sizes = [int((labels == c).sum()) for c in range(k)]
    min_sz = min(sizes)
    j_mean, j_std = bootstrap_jaccard_stability(
        X3, k, N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE
    )
    stability = 'STABLE' if j_mean >= 0.75 else 'UNSTABLE'
    results_3.append(dict(
        k=k, silhouette=round(sil,4), CH=round(ch,1),
        DB=round(db,4), min_size=min_sz, cluster_sizes=sizes,
        jaccard_mean=round(j_mean,3), jaccard_std=round(j_std,3),
        stability=stability
    ))
    flag = ' ✓ min>=30' if min_sz >= MIN_CLUSTER else ' ✗ min<30'
    print(f'k={k}: sil={sil:.3f} CH={ch:.0f} DB={db:.3f}  '
          f'min_size={min_sz}{flag}  Jaccard={j_mean:.3f}±{j_std:.3f} ({stability})')
    print(f'       sizes: {sizes}')

print()

# Select best k: highest silhouette among valid k (all clusters >= 30)
valid_k3 = [r for r in results_3 if r['min_size'] >= MIN_CLUSTER]
if valid_k3:
    best_3 = max(valid_k3, key=lambda r: r['silhouette'])
    print(f'Selected k (3-feature, valid): {best_3["k"]} '
          f'(silhouette={best_3["silhouette"]}, all clusters >= {MIN_CLUSTER})')
else:
    best_3 = next(r for r in results_3 if r['k'] == 2)
    print(f'FALLBACK: No k had all clusters >= {MIN_CLUSTER}. Using k=2.')
    print(f'  k=2: silhouette={best_3["silhouette"]}, sizes={best_3["cluster_sizes"]}')

SELECTED_K3 = best_3['k']

In [ ]:
# ─── 4c. Figure A4: Cluster selection diagnostics (3-feature) ─────────────────

ks = [r['k'] for r in results_3]
sils = [r['silhouette'] for r in results_3]
chs  = [r['CH'] for r in results_3]
dbs  = [r['DB'] for r in results_3]
jacs = [r['jaccard_mean'] for r in results_3]
j_stds = [r['jaccard_std'] for r in results_3]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('DS5 Path A — Cluster selection diagnostics (core 3-feature set)', fontsize=13)

axes[0,0].plot(ks, sils, 'o-', color='steelblue', linewidth=2)
axes[0,0].axvline(SELECTED_K3, color='red', linestyle='--', linewidth=1.2, label=f'selected k={SELECTED_K3}')
axes[0,0].set_title('Silhouette score (higher = better)')
axes[0,0].set_xlabel('k'); axes[0,0].set_ylabel('Silhouette')
axes[0,0].legend()

axes[0,1].plot(ks, chs, 'o-', color='darkorange', linewidth=2)
axes[0,1].axvline(SELECTED_K3, color='red', linestyle='--', linewidth=1.2)
axes[0,1].set_title('Calinski-Harabasz (higher = better)')
axes[0,1].set_xlabel('k'); axes[0,1].set_ylabel('CH score')

axes[1,0].plot(ks, dbs, 'o-', color='seagreen', linewidth=2)
axes[1,0].axvline(SELECTED_K3, color='red', linestyle='--', linewidth=1.2)
axes[1,0].set_title('Davies-Bouldin (lower = better)')
axes[1,0].set_xlabel('k'); axes[1,0].set_ylabel('DB score')

axes[1,1].errorbar(ks, jacs, yerr=j_stds, fmt='o-', color='purple', linewidth=2, capsize=5)
axes[1,1].axhline(0.75, color='gray', linestyle='--', linewidth=1.0, label='stability threshold (0.75)')
axes[1,1].axvline(SELECTED_K3, color='red', linestyle='--', linewidth=1.2)
axes[1,1].set_title('Bootstrap Jaccard stability (higher = better)')
axes[1,1].set_xlabel('k'); axes[1,1].set_ylabel('Jaccard (mean ± std)')
axes[1,1].legend()

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a4_cluster_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a4_cluster_selection.png')

## 5. Step 4 — Sensitivity run with H_hour

In [ ]:
# ─── 5. Sensitivity run: 4-feature (adds H_hour) ──────────────────────────────

results_4 = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X4)
    sil  = silhouette_score(X4, labels)
    ch   = calinski_harabasz_score(X4, labels)
    db   = davies_bouldin_score(X4, labels)
    sizes = [int((labels == c).sum()) for c in range(k)]
    j_mean, j_std = bootstrap_jaccard_stability(
        X4, k, N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE
    )
    stability = 'STABLE' if j_mean >= 0.75 else 'UNSTABLE'
    results_4.append(dict(
        k=k, silhouette=round(sil,4), CH=round(ch,1),
        DB=round(db,4), min_size=min(sizes),
        cluster_sizes=sizes,
        jaccard_mean=round(j_mean,3), jaccard_std=round(j_std,3),
        stability=stability
    ))
    print(f'k={k}: sil={sil:.3f} CH={ch:.0f} DB={db:.3f}  '
          f'min_size={min(sizes)}  Jaccard={j_mean:.3f}±{j_std:.3f} ({stability})')

print()

# Select best k for sensitivity
valid_k4 = [r for r in results_4 if r['min_size'] >= MIN_CLUSTER]
if valid_k4:
    best_4 = max(valid_k4, key=lambda r: r['silhouette'])
else:
    best_4 = next(r for r in results_4 if r['k'] == 2)
    print(f'FALLBACK k=2 for 4-feature sensitivity.')

SELECTED_K4 = best_4['k']
print(f'Selected k (4-feature sensitivity): {SELECTED_K4}')

# ARI between 3-feature and 4-feature solutions
km3 = KMeans(n_clusters=SELECTED_K3, init='k-means++', n_init=10, random_state=RANDOM_STATE)
km4 = KMeans(n_clusters=SELECTED_K4, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_3_final = km3.fit_predict(X3)
labels_4_final = km4.fit_predict(X4)

ari_3v4 = adjusted_rand_score(labels_3_final, labels_4_final)
print(f'\nARI(3-feature k={SELECTED_K3} vs 4-feature k={SELECTED_K4}): {ari_3v4:.3f}')
if ari_3v4 > 0.90:
    print('  → H_hour does NOT change the cluster structure materially.')
elif ari_3v4 > 0.60:
    print('  → H_hour partially reorganizes clusters; moderate effect.')
else:
    print('  → H_hour substantially changes cluster assignments; report as different solution.')

## 6. Step 5 — Final cluster labels and profiles

In [ ]:
# ─── 6a. Assign final cluster labels (3-feature primary solution) ──────────────

# Run final k-means with selected k
km_final = KMeans(n_clusters=SELECTED_K3, init='k-means++', n_init=10,
                  random_state=RANDOM_STATE)
raw_labels = km_final.fit_predict(X3)

# Order clusters by mean sig_hat ascending (C1 = lowest sig_hat / most stable)
cluster_sig_means = [
    (c, df['sig_hat'].values[raw_labels == c].mean())
    for c in range(SELECTED_K3)
]
cluster_sig_means.sort(key=lambda x: x[1])
label_map = {orig: new for new, (orig, _) in enumerate(cluster_sig_means)}
labels_ordered = np.array([label_map[l] for l in raw_labels])

# Create named labels: C1, C2, ...
cluster_name_map = {i: f'C{i+1}' for i in range(SELECTED_K3)}
df['cluster_behav'] = pd.Categorical(
    [cluster_name_map[l] for l in labels_ordered]
)

print('=== Final cluster sizes (3-feature primary) ===')
print(df['cluster_behav'].value_counts().sort_index())
print()

# Retrieve selected k stability info
sel_info = next(r for r in results_3 if r['k'] == SELECTED_K3)
print(f'Silhouette: {sel_info["silhouette"]}')
print(f'Jaccard: {sel_info["jaccard_mean"]} ± {sel_info["jaccard_std"]} ({sel_info["stability"]})')

In [ ]:
# ─── 6b. Cluster profiles on all features ─────────────────────────────────────

PROFILE_FEATURES = (
    CORE_FEATURES +
    ['H_hour', 'mu_hat', 'sigma_mu_ratio', 'H_out_resid', 'chi_behav',
     'HHI_net', 'rho_self', 'H_trans', 'H_app']  # NeT as overlay
)

print('=== Cluster profiles: mean and median ===')
for feat_name in PROFILE_FEATURES:
    if feat_name not in df.columns:
        continue
    row = []
    for cname in sorted(df['cluster_behav'].unique()):
        vals = df.loc[df['cluster_behav'] == cname, feat_name].dropna()
        row.append(f'{cname}: mean={vals.mean():.4f} med={vals.median():.4f} (N={len(vals)})')
    print(f'  {feat_name}: ' + ' | '.join(row))

print()
print('=== Condition distribution ===')
ctab = pd.crosstab(df['cluster_behav'], df['study_condition'])
print(ctab)
chi2, p_cond, dof, _ = chi2_contingency(ctab)
print(f'chi2={chi2:.3f}, p={p_cond:.4f} (df={dof})')
if p_cond > 0.05:
    print('  → Condition-independent ✓')
else:
    print('  ⚠ Cluster distribution differs by condition — interpret carefully')

In [ ]:
# ─── 6c. Psychosocial Kruskal-Wallis tests (Bonferroni correction) ────────────

PSY_SCALES = [
    'fomo_mean', 'belong_mean', 'ostracism_mean',
    'digital_wellbeing', 'social_media_wellbeing'
]
n_tests = len(PSY_SCALES)

print('=== Psychosocial Kruskal-Wallis tests (Bonferroni-corrected) ===')
psy_results = []
for scale in PSY_SCALES:
    if scale not in df.columns:
        continue
    groups = [df.loc[df['cluster_behav'] == c, scale].dropna().values
              for c in sorted(df['cluster_behav'].unique())]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = kruskal(*groups)
    p_bonf = min(p_val * n_tests, 1.0)
    sig = '* SIGNIFICANT' if p_bonf < 0.05 else ''
    psy_results.append(dict(scale=scale, H=round(h_stat,3), p=round(p_val,4),
                            p_bonf=round(p_bonf,4)))
    print(f'  {scale}: H={h_stat:.3f}, p={p_val:.4f}, p_bonf={p_bonf:.4f} {sig}')

n_sig = sum(1 for r in psy_results if r['p_bonf'] < 0.05)
print(f'\n{n_sig}/{len(psy_results)} psychosocial scales significant after Bonferroni.')

In [ ]:
# ─── 6d. Figure A5: Cluster profiles bar chart ────────────────────────────────

profile_summary = df.groupby('cluster_behav')[CORE_FEATURES + ['H_hour']].mean()
cluster_names = profile_summary.index.tolist()

fig, axes = plt.subplots(1, len(CORE_FEATURES) + 1, figsize=(14, 5), sharey=False)
fig.suptitle('DS5 Path A — Behavior-only cluster profiles (means)',
             fontsize=13)

colors = plt.cm.Set2(np.linspace(0, 0.8, SELECTED_K3))
features_to_plot = CORE_FEATURES + ['H_hour']

for ax, feat_name in zip(axes, features_to_plot):
    vals = [profile_summary.loc[c, feat_name] for c in cluster_names]
    bars = ax.bar(cluster_names, vals, color=colors[:len(cluster_names)])
    ax.set_title(feat_name, fontsize=10)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Mean')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a5_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a5_cluster_profiles.png')

In [ ]:
# ─── 6e. Figure A6: Overlay features (mu_hat, sigma_mu_ratio, H_out_resid, cond) ─

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('DS5 Path A — Cluster overlay features', fontsize=13)

# Panel 1: mu_hat by cluster (violin)
ax = axes[0, 0]
mu_data = [df.loc[df['cluster_behav'] == c, 'mu_hat'].dropna().values
           for c in sorted(df['cluster_behav'].unique())]
ax.violinplot(mu_data, positions=range(len(cluster_names)), showmedians=True)
ax.set_xticks(range(len(cluster_names)))
ax.set_xticklabels(sorted(df['cluster_behav'].unique()))
ax.set_title('mu_hat (OU drift) — post-hoc overlay')
ax.set_ylabel('mu_hat')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.text(0.02, 0.02, 'NOTE: 56.5% near-zero; use with caution',
        transform=ax.transAxes, fontsize=7, color='gray')

# Panel 2: sigma_mu_ratio by cluster (violin; subset only)
ax = axes[0, 1]
smr_data = [df.loc[(df['cluster_behav'] == c) & df['sigma_mu_ratio'].notna(),
                   'sigma_mu_ratio'].values
            for c in sorted(df['cluster_behav'].unique())]
smr_data_nonempty = [d for d in smr_data if len(d) > 0]
if len(smr_data_nonempty) > 0:
    ax.violinplot(smr_data_nonempty, positions=range(len(smr_data_nonempty)), showmedians=True)
    ax.set_xticks(range(len(cluster_names)))
    ax.set_xticklabels(sorted(df['cluster_behav'].unique()))
ax.set_title('sigma_mu_ratio — post-hoc overlay (31% NaN subset)')
ax.set_ylabel('sigma_mu_ratio')

# Panel 3: H_out_resid by cluster
ax = axes[1, 0]
if 'H_out_resid' in df.columns:
    hor_data = [df.loc[df['cluster_behav'] == c, 'H_out_resid'].dropna().values
                for c in sorted(df['cluster_behav'].unique())]
    hor_data_nonempty = [d for d in hor_data if len(d) > 0]
    if hor_data_nonempty:
        ax.violinplot(hor_data_nonempty,
                      positions=range(len(hor_data_nonempty)), showmedians=True)
        ax.set_xticks(range(len(cluster_names)))
        ax.set_xticklabels(sorted(df['cluster_behav'].unique()))
ax.set_title('H_out_resid (condition-residualized friction outcome entropy)')
ax.set_ylabel('H_out_resid')

# Panel 4: Condition distribution stacked bar
# Use explicit color list to avoid mpl.colormaps compatibility issue
ax = axes[1, 1]
cond_tab = pd.crosstab(df['cluster_behav'], df['study_condition'], normalize='index') * 100
cond_colors = ['#8dd3c7', '#ffffb3', '#bebada'][:len(cond_tab.columns)]
cond_tab.plot(kind='bar', stacked=True, ax=ax,
              color=cond_colors, edgecolor='white')
ax.set_title(f'Condition distribution\nchi2={chi2:.2f}, p={p_cond:.4f}')
ax.set_ylabel('% of cluster')
ax.set_xlabel('Behavior-only cluster')
ax.legend(title='Condition', fontsize=8, bbox_to_anchor=(1.0, 1.0))
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a6_cluster_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a6_cluster_overlay.png')

## 7. Step 6 — Cross-path comparison: Path A vs Phase 1 and Path C

In [ ]:
# ─── 7a. ARI comparisons ─────────────────────────────────────────────────────

# Comparison 1: Path A vs Phase 1 profile
mask_p1 = df['profile_p1'].notna() & df['cluster_behav'].notna()
ari_vs_p1 = adjusted_rand_score(
    df.loc[mask_p1, 'profile_p1'],
    df.loc[mask_p1, 'cluster_behav']
)
print(f'ARI(Path A cluster vs Phase 1 profile): {ari_vs_p1:.3f}')

if ari_vs_p1 > 0.50:
    print('  → HIGH overlap: behavior-only mostly reproduces the NeT-driven split')
    print('     (behavioral features are correlated with network type despite no NeT input)')
elif ari_vs_p1 > 0.20:
    print('  → MODERATE overlap: partial correspondence with Phase 1 profile')
else:
    print('  → LOW overlap: behavior-only reveals a genuinely orthogonal structure')
    print('     (behavioral dynamics and network topology are independent dimensions)')

print()

# Comparison 2: Path A vs Path C sub-cluster composite
mask_C = df['subprofile_C'].notna() & df['cluster_behav'].notna()
ari_vs_C = adjusted_rand_score(
    df.loc[mask_C, 'subprofile_C'],
    df.loc[mask_C, 'cluster_behav']
)
print(f'ARI(Path A cluster vs Path C sub-clusters [B1/B2/A1/A2]): {ari_vs_C:.3f}')

print()

# Comparison 3: Path A vs NeT-only full-sample cluster
mask_net = df['cluster_net_only'].notna() & df['cluster_behav'].notna()
ari_vs_net = adjusted_rand_score(
    df.loc[mask_net, 'cluster_net_only'],
    df.loc[mask_net, 'cluster_behav']
)
print(f'ARI(Path A cluster vs NeT-only full-sample cluster): {ari_vs_net:.3f}')
print()

print('=== Summary: Path A orthogonality ===')
print(f'  vs Phase 1:       ARI = {ari_vs_p1:.3f}')
print(f'  vs Path C (B1/B2/A1/A2): ARI = {ari_vs_C:.3f}')
print(f'  vs NeT-only:      ARI = {ari_vs_net:.3f}')

In [ ]:
# ─── 7b. Contingency table: Path A cluster × Phase 1 profile ──────────────────

print('=== Contingency table: Path A cluster × Phase 1 profile ===')
ctab_p1 = pd.crosstab(
    df['cluster_behav'], df['profile_p1'],
    margins=True
)
print(ctab_p1)
print()

# Row-normalised (% within each Path A cluster)
print('=== Row-normalized (% Phase 1 profile within each Path A cluster) ===')
print(pd.crosstab(df['cluster_behav'], df['profile_p1'], normalize='index').round(3) * 100)

print()
print('=== Contingency table: Path A cluster × Path C subprofile ===')
ctab_C = pd.crosstab(
    df.loc[df['subprofile_C'].notna(), 'cluster_behav'],
    df.loc[df['subprofile_C'].notna(), 'subprofile_C'],
    margins=True
)
print(ctab_C)

In [ ]:
# ─── 7c. Figure A7: Cross-path comparison stacked bar ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'DS5 Path A — Cross-path comparison\n'
             f'ARI(vs Phase 1)={ari_vs_p1:.3f}  |  '
             f'ARI(vs Path C)={ari_vs_C:.3f}  |  '
             f'ARI(vs NeT-only)={ari_vs_net:.3f}',
             fontsize=12)

# Panel 1: Path A cluster × Phase 1 profile (stacked bar, count)
ctab_plot = pd.crosstab(df['cluster_behav'], df['profile_p1'])
ctab_plot.plot(kind='bar', stacked=True, ax=axes[0],
               color=['#e67e22', '#2980b9'], edgecolor='white')
axes[0].set_title('Path A cluster × Phase 1 profile')
axes[0].set_xlabel('Behavior-only cluster (Path A)')
axes[0].set_ylabel('N participants')
axes[0].legend(title='Phase 1 profile')
axes[0].tick_params(axis='x', rotation=0)

# Panel 2: Path A cluster × Path C subprofile (stacked bar, count)
# Explicit colors to avoid mpl.colormaps compatibility issue with old matplotlib
set2_colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
df_c_sub = df[df['subprofile_C'].notna()].copy()
ctab_c_plot = pd.crosstab(df_c_sub['cluster_behav'], df_c_sub['subprofile_C'])
ctab_c_plot.plot(kind='bar', stacked=True, ax=axes[1],
                 color=set2_colors[:len(ctab_c_plot.columns)], edgecolor='white')
axes[1].set_title('Path A cluster × Path C sub-profiles (B1/B2/A1/A2)')
axes[1].set_xlabel('Behavior-only cluster (Path A)')
axes[1].set_ylabel('N participants')
axes[1].legend(title='Path C subprofile')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_a7_crosspath_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_a7_crosspath_comparison.png')

## 8. Save outputs

In [ ]:
# ─── 8a. Build cluster assignments CSV ─────────────────────────────────────────

assignments_out = df[[
    'participant_id_canonical', 'study_condition',
    'profile_p1',         # Phase 1 label (post-hoc)
    'subprofile_C',       # Path C composite label (post-hoc)
    'cluster_net_only',   # Path C NeT-only label (post-hoc)
    'cluster_behav'       # PATH A RESULT
]].copy()

assignments_out.to_csv(PHENOTYPING / 'ds5_behav_only_cluster_assignments.csv', index=False)
print(f'Saved: ds5_behav_only_cluster_assignments.csv ({len(assignments_out)} rows)')
print(assignments_out.head())

In [ ]:
# ─── 8b. Build cluster profiles CSV ──────────────────────────────────────────

PROFILE_COLS = (
    CORE_FEATURES +
    ['H_hour', 'mu_hat', 'sigma_mu_ratio', 'H_out_resid', 'chi_behav',
     'HHI_net', 'rho_self', 'H_trans', 'H_app']
)
PROFILE_COLS = [c for c in PROFILE_COLS if c in df.columns]

rows = []
for cname in sorted(df['cluster_behav'].unique()):
    grp = df[df['cluster_behav'] == cname]
    row = {'cluster': cname, 'N': len(grp)}
    for col in PROFILE_COLS:
        vals = grp[col].dropna()
        row[f'{col}_mean']   = round(vals.mean(), 4) if len(vals) > 0 else np.nan
        row[f'{col}_median'] = round(vals.median(), 4) if len(vals) > 0 else np.nan
        row[f'{col}_std']    = round(vals.std(), 4) if len(vals) > 0 else np.nan
    rows.append(row)

profiles_out = pd.DataFrame(rows)
profiles_out.to_csv(PHENOTYPING / 'ds5_behav_only_cluster_profiles.csv', index=False)
print(f'Saved: ds5_behav_only_cluster_profiles.csv ({len(profiles_out)} rows)')
print(profiles_out[['cluster', 'N'] +
                    [f'{c}_mean' for c in CORE_FEATURES + ['H_hour']]].to_string())

In [ ]:
# ─── 8c. Build stability CSV ─────────────────────────────────────────────────

stability_rows = []
for r in results_3:
    stability_rows.append(dict(
        analysis='3-feature core (sig_hat, H_emp, x_mean)',
        k=r['k'], silhouette=r['silhouette'],
        calinski_harabasz=r['CH'], davies_bouldin=r['DB'],
        jaccard_mean=r['jaccard_mean'], jaccard_std=r['jaccard_std'],
        stability=r['stability'],
        cluster_sizes=str(r['cluster_sizes']),
        selected=(r['k'] == SELECTED_K3)
    ))
for r in results_4:
    stability_rows.append(dict(
        analysis='4-feature sensitivity (+ H_hour)',
        k=r['k'], silhouette=r['silhouette'],
        calinski_harabasz=r['CH'], davies_bouldin=r['DB'],
        jaccard_mean=r['jaccard_mean'], jaccard_std=r['jaccard_std'],
        stability=r['stability'],
        cluster_sizes=str(r['cluster_sizes']),
        selected=(r['k'] == SELECTED_K4)
    ))

stability_out = pd.DataFrame(stability_rows)
stability_out.to_csv(PHENOTYPING / 'ds5_behav_only_cluster_stability.csv', index=False)
print(f'Saved: ds5_behav_only_cluster_stability.csv ({len(stability_out)} rows)')
print(stability_out[['analysis','k','silhouette','jaccard_mean','jaccard_std','stability','selected']].to_string())

In [ ]:
# ─── 8d. Write model_run_summary_path_a.md ────────────────────────────────────

sel_3 = next(r for r in results_3 if r['k'] == SELECTED_K3)
sel_4 = next(r for r in results_4 if r['k'] == SELECTED_K4)

summary_text = f"""# DS5 Step 2 Path A — Behavior-Only Phenotyping: Model Run Summary

**Date:** 2026-04-19
**Notebook:** 102_ds5_behavior_only_phenotype.ipynb
**Stage:** Stage 4, Step 2 — Path A

---

## Run parameters
- Core clustering features: {CORE_FEATURES}
- Sensitivity features: {SENS_FEATURES}
- RANDOM_STATE: {RANDOM_STATE}
- Bootstrap: {N_BOOTSTRAPS} iterations, {SAMPLE_FRAC*100:.0f}% sample fraction
- Min cluster size threshold: {MIN_CLUSTER}
- Analysis population: N = {len(df)}

## Feature notes
- r(H_hour, sig_hat) = {df['H_hour'].corr(df['sig_hat']):.3f} — strong negative; H_hour excluded from primary clustering
- mu_hat near-zero (|<0.01|): {(df['mu_hat'].abs()<0.01).mean()*100:.1f}% — overlay only
- sigma_mu_ratio NaN: {df['sigma_mu_ratio'].isna().sum()} of {len(df)} ({df['sigma_mu_ratio'].isna().mean()*100:.1f}%) — overlay only

## Primary clustering (3-feature core)
- Selected k: {SELECTED_K3}
- Silhouette: {sel_3['silhouette']}
- Bootstrap Jaccard: {sel_3['jaccard_mean']} ± {sel_3['jaccard_std']} ({sel_3['stability']})
- Cluster sizes: {sel_3['cluster_sizes']}
- Condition χ²: {chi2:.3f}, p={p_cond:.4f}
- Psychosocial significant (after Bonferroni): {n_sig} of {len(psy_results)}

## Cluster profiles (primary solution)
"""

for cname in sorted(df['cluster_behav'].unique()):
    grp = df[df['cluster_behav'] == cname]
    summary_text += f"""
### {cname} (N={len(grp)})
- sig_hat: mean={grp['sig_hat'].mean():.4f}, median={grp['sig_hat'].median():.4f}
- H_emp: mean={grp['H_emp'].mean():.4f}, median={grp['H_emp'].median():.4f}
- x_mean: mean={grp['x_mean'].mean():.4f}, median={grp['x_mean'].median():.4f}
- H_hour: mean={grp['H_hour'].mean():.4f}, median={grp['H_hour'].median():.4f}
- HHI_net (overlay): mean={grp['HHI_net'].mean():.4f}
- rho_self (overlay): mean={grp['rho_self'].mean():.4f}
"""

summary_text += f"""
## Sensitivity run (4-feature: adds H_hour)
- Selected k: {SELECTED_K4}
- Silhouette: {sel_4['silhouette']}
- Bootstrap Jaccard: {sel_4['jaccard_mean']} ± {sel_4['jaccard_std']} ({sel_4['stability']})
- ARI(3-feature vs 4-feature): {ari_3v4:.3f}

## Cross-path comparison
- ARI(Path A vs Phase 1 profile):          {ari_vs_p1:.3f}
- ARI(Path A vs Path C subprofiles):       {ari_vs_C:.3f}
- ARI(Path A vs NeT-only full-sample):     {ari_vs_net:.3f}

## Output files
- ds5_behav_only_cluster_assignments.csv
- ds5_behav_only_cluster_profiles.csv
- ds5_behav_only_cluster_stability.csv
- figures/fig_a1_behav_only_distributions.png
- figures/fig_a2_behav_pairplot.png
- figures/fig_a3_behav_pca.png
- figures/fig_a4_cluster_selection.png
- figures/fig_a5_cluster_profiles.png
- figures/fig_a6_cluster_overlay.png
- figures/fig_a7_crosspath_comparison.png

## Key limitations
1. sig_hat, H_emp, x_mean may still correlate with network type despite no NeT input
2. mu_hat is near-zero for most participants — no drift dimension in DS5
3. sigma_mu_ratio 31% NaN — impulsivity ratio not available for full sample
4. H_hour strongly correlated with sig_hat (r={df['H_hour'].corr(df['sig_hat']):.3f}) — \
not independent
5. All findings are exploratory and DS5-specific; cross-dataset replication required
"""

with open(PHENOTYPING / 'model_run_summary_path_a.md', 'w') as f:
    f.write(summary_text)

print('Saved: model_run_summary_path_a.md')

In [ ]:
# ─── 8e. Write companion .md files ────────────────────────────────────────────

# --- ds5_behav_only_cluster_assignments.md ---
sel_3_info = next(r for r in results_3 if r['k'] == SELECTED_K3)
cluster_descs = []
for cname in sorted(df['cluster_behav'].unique()):
    grp = df[df['cluster_behav'] == cname]
    cluster_descs.append(
        f'| {cname} | {len(grp)} | '
        f'sig_hat={grp["sig_hat"].mean():.3f}, '
        f'H_emp={grp["H_emp"].mean():.3f}, '
        f'x_mean={grp["x_mean"].mean():.3f} |'
    )

# Pre-compute expressions to avoid backslash-in-f-string (Python 3.9 limitation)
_cluster_table = '\n'.join(cluster_descs)
_ari_p1_label = ('High' if ari_vs_p1 > 0.5 else 'Moderate' if ari_vs_p1 > 0.2 else 'Low')
_ari_p1_interp = ('behavior reproduces NeT split' if ari_vs_p1 > 0.5
                  else 'partial correspondence' if ari_vs_p1 > 0.2
                  else 'orthogonal structure')
_ari_net_label = ('High' if ari_vs_net > 0.5 else 'Low')

assignments_md = (
    f"# ds5_behav_only_cluster_assignments.csv — Output Documentation\n\n"
    f"**Produced by:** `notebooks/102_ds5_behavior_only_phenotype.ipynb`\n"
    f"**Stage:** Stage 4, Step 2 — Path A (Behavior-Only Phenotyping)\n"
    f"**Date:** 2026-04-19\n"
    f"**Blueprint:** `outputs/reports/stage4/ds5_path_a_behavior_only_blueprint.md`\n\n"
    f"---\n\n"
    f"## What this file contains\n\n"
    f"Behavior-only cluster assignments for DS5 participants from Path A. Each row is one\n"
    f"participant. Cluster labels are derived from k-means on [sig_hat, H_emp, x_mean] with\n"
    f"NO network topology features. Phase 1 and Path C labels are included for post-hoc\n"
    f"cross-path comparison.\n\n"
    f"**Unit of observation:** One participant (participant_id_canonical)\n\n"
    f"---\n\n"
    f"## Column reference\n\n"
    f"| Column | Type | Description |\n"
    f"|---|---|---|\n"
    f"| `participant_id_canonical` | str | Unique participant identifier |\n"
    f"| `study_condition` | str | Experimental condition: control / default / reintervention |\n"
    f"| `profile_p1` | str | Phase 1 cluster label (post-hoc overlay, not used in clustering) |\n"
    f"| `subprofile_C` | str | Path C composite sub-cluster label (B1/B2/A1/A2; post-hoc overlay) |\n"
    f"| `cluster_net_only` | int | Path C NeT-only cluster index (post-hoc overlay) |\n"
    f"| `cluster_behav` | str | **PATH A RESULT** — Behavior-only cluster label (C1, C2, ...) |\n\n"
    f"---\n\n"
    f"## Cluster definitions (k={SELECTED_K3} primary solution)\n\n"
    f"**Clustering features:** sig_hat, H_emp, x_mean (StandardScaler; fit on N={len(df)})\n"
    f"**Bootstrap Jaccard:** {sel_3_info['jaccard_mean']} ± {sel_3_info['jaccard_std']} ({sel_3_info['stability']})\n"
    f"**Silhouette:** {sel_3_info['silhouette']}\n\n"
    f"Clusters ordered by mean sig_hat ascending (C1 = lowest noise).\n\n"
    f"| Cluster | N | Key means |\n"
    f"|---|---|---|\n"
    f"{_cluster_table}\n\n"
    f"---\n\n"
    f"## Cross-path ARI summary\n\n"
    f"| Comparison | ARI | Interpretation |\n"
    f"|---|---|---|\n"
    f"| Path A vs Phase 1 profile | {ari_vs_p1:.3f} | {_ari_p1_label} overlap — {_ari_p1_interp} |\n"
    f"| Path A vs Path C subprofiles | {ari_vs_C:.3f} | See full interpretation in run summary |\n"
    f"| Path A vs NeT-only cluster | {ari_vs_net:.3f} | {_ari_net_label} overlap |\n\n"
    f"---\n\n"
    f"## How to load\n\n"
    f"```python\n"
    f"import pandas as pd\n"
    f"from pathlib import Path\n\n"
    f"BASE = Path('outputs/model_outputs/phenotyping')\n"
    f"assignments = pd.read_csv(BASE / 'ds5_behav_only_cluster_assignments.csv')\n\n"
    f"# Behavior-only cluster\n"
    f"behav_labels = assignments[['participant_id_canonical', 'cluster_behav']]\n"
    f"```\n\n"
    f"---\n\n"
    f"## Key limitations\n\n"
    f"1. sig_hat, H_emp, x_mean may still correlate with network type even without NeT input\n"
    f"2. ARI vs Phase 1 = {ari_vs_p1:.3f} — interpret with care (see run summary)\n"
    f"3. All findings are exploratory and DS5-specific; not validated across datasets\n"
)

with open(PHENOTYPING / 'ds5_behav_only_cluster_assignments.md', 'w') as f:
    f.write(assignments_md)
print('Saved: ds5_behav_only_cluster_assignments.md')

# --- ds5_behav_only_cluster_profiles.md ---
profile_table_rows = []
for cname in sorted(df['cluster_behav'].unique()):
    grp = df[df['cluster_behav'] == cname]
    profile_table_rows.append(
        f'| {cname} | {len(grp)} | '
        f'{grp["sig_hat"].mean():.4f} | {grp["H_emp"].mean():.4f} | '
        f'{grp["x_mean"].mean():.4f} | {grp["H_hour"].mean():.4f} | '
        f'{grp["HHI_net"].mean():.4f} | {grp["rho_self"].mean():.4f} |'
    )

_profile_table = '\n'.join(profile_table_rows)

profiles_md = (
    f"# ds5_behav_only_cluster_profiles.csv — Output Documentation\n\n"
    f"**Produced by:** `notebooks/102_ds5_behavior_only_phenotype.ipynb`\n"
    f"**Stage:** Stage 4, Step 2 — Path A (Behavior-Only Phenotyping)\n"
    f"**Date:** 2026-04-19\n\n"
    f"---\n\n"
    f"## What this file contains\n\n"
    f"Per-cluster feature profiles (mean, median, std) for all behavior-only clusters found in\n"
    f"Path A. Rows represent clusters; columns represent feature statistics.\n\n"
    f"**Unit of observation:** One cluster\n\n"
    f"---\n\n"
    f"## Column reference (abbreviated)\n\n"
    f"| Column pattern | Description |\n"
    f"|---|---|\n"
    f"| `cluster` | Cluster label (C1, C2, ...) |\n"
    f"| `N` | Number of participants in cluster |\n"
    f"| `{{feature}}_mean` | Mean of feature across cluster participants |\n"
    f"| `{{feature}}_median` | Median of feature across cluster participants |\n"
    f"| `{{feature}}_std` | Standard deviation of feature across cluster participants |\n\n"
    f"Features included: sig_hat, H_emp, x_mean, H_hour, mu_hat, sigma_mu_ratio,\n"
    f"H_out_resid, chi_behav, HHI_net (overlay), rho_self (overlay), H_trans (overlay),\n"
    f"H_app (overlay)\n\n"
    f"---\n\n"
    f"## Cluster summary (key features)\n\n"
    f"| Cluster | N | sig_hat | H_emp | x_mean | H_hour | HHI_net (overlay) | rho_self (overlay) |\n"
    f"|---|---|---|---|---|---|---|---|\n"
    f"{_profile_table}\n\n"
    f"---\n\n"
    f"## Interpretation\n\n"
    f"Clusters ordered by sig_hat ascending (C1 = lowest noise / most stable behavior).\n"
    f"NeT features (HHI_net, rho_self) are post-hoc overlays — NOT used in clustering.\n"
    f"ARI between Path A clusters and Phase 1 profile = {ari_vs_p1:.3f} (see assignments doc).\n\n"
    f"---\n\n"
    f"## How to load\n\n"
    f"```python\n"
    f"import pandas as pd\n"
    f"from pathlib import Path\n\n"
    f"BASE = Path('outputs/model_outputs/phenotyping')\n"
    f"profiles = pd.read_csv(BASE / 'ds5_behav_only_cluster_profiles.csv')\n"
    f"```\n"
)

with open(PHENOTYPING / 'ds5_behav_only_cluster_profiles.md', 'w') as f:
    f.write(profiles_md)
print('Saved: ds5_behav_only_cluster_profiles.md')

print()
print('=== All Path A outputs saved successfully ===')

In [ ]:
# ─── Final summary ────────────────────────────────────────────────────────────

print('=== DS5 Step 2 Path A — COMPLETE ===')
print()
print(f'Selected solution: k={SELECTED_K3} (3-feature: sig_hat, H_emp, x_mean)')
print(f'  Silhouette: {sel_3_info["silhouette"]}')
print(f'  Jaccard: {sel_3_info["jaccard_mean"]} ± {sel_3_info["jaccard_std"]} ({sel_3_info["stability"]})')
print()
print(f'Cluster sizes: {sel_3_info["cluster_sizes"]}')
print()
print('Cross-path orthogonality:')
print(f'  ARI vs Phase 1:        {ari_vs_p1:.3f}')
print(f'  ARI vs Path C subtypes: {ari_vs_C:.3f}')
print(f'  ARI vs NeT-only:       {ari_vs_net:.3f}')
print()
print(f'H_hour sensitivity: ARI(3-feature vs 4-feature) = {ari_3v4:.3f}')
print()
print('Files saved:')
for fname in [
    'ds5_behav_only_cluster_assignments.csv',
    'ds5_behav_only_cluster_assignments.md',
    'ds5_behav_only_cluster_profiles.csv',
    'ds5_behav_only_cluster_profiles.md',
    'ds5_behav_only_cluster_stability.csv',
    'model_run_summary_path_a.md',
]:
    exists = (PHENOTYPING / fname).exists()
    print(f'  {fname}: {"✓" if exists else "MISSING"}')
for fname in [
    'fig_a1_behav_only_distributions.png',
    'fig_a2_behav_pairplot.png',
    'fig_a3_behav_pca.png',
    'fig_a4_cluster_selection.png',
    'fig_a5_cluster_profiles.png',
    'fig_a6_cluster_overlay.png',
    'fig_a7_crosspath_comparison.png',
]:
    exists = (FIG_DIR / fname).exists()
    print(f'  figures/{fname}: {"✓" if exists else "MISSING"}')